In [1]:
import os

from pathlib import Path
os.chdir(Path.cwd().parent)
import time
import copy
import json
import random
import warnings
import numpy as np
from pathlib import Path
from collections import defaultdict
from IPython.core.magic import register_cell_magic

from data.dataloader import load_dataset
from utils.utils import set_random_seed, Usage
from utils.llm import load_model
from utils.metrics import get_metrics_fn
from utils.multiprocess import multiprocess_call
from modules.post_process import get_label_tokenizer, get_logit, prepare_labels, match_condition_fuzz
from TherapyReason.significance import get_significance
from TherapyReason.therapy import get_dl_candidates, get_kg_candidates, get_therapy, rank_dl_candidates
from TherapyReason.overview import process_solution_overview_wrap
from TherapyReason.reasoning import candidate2string_diag, template_solution_dl_diag, solution2string_with_overview_diag, template_solution_system_diag, \
    post_process_solution_wrapper_diag, template_solution_kg_diag, candidate_similarity
from modules.context import get_concepts

set_random_seed(42)
dataset = 'mimic3'
task = 'diagnosis'
label_key = 'label'
datasets, full_set = load_dataset(dataset, task)
test_set = datasets['test']
label_tokenizer = get_label_tokenizer(full_set, label_key)
cond_tokenizer = get_label_tokenizer(full_set, 'conditions')
proc_tokenizer = get_label_tokenizer(full_set, 'procedures')
drug_tokenizer = get_label_tokenizer(full_set, 'drugs')
significance = get_significance(dataset, task)
cache_dir = f"./TherapyReason/cache/{dataset}-{task}"
os.makedirs(cache_dir, exist_ok=True)
concept_similarity = json.loads(Path("TherapyReason/cache/llm_pairwise.json").read_text())
print(len(test_set))

warnings.filterwarnings('ignore')


@register_cell_magic
def skip(line, cell): return None

# Qwen3-235B-A22B-Instruct-2507-ideaLAB
# Qwen3-30B-A3B
# DeepSeek-V3-671B
llm = load_model({"model": "deepseek-chat", "server": "deepseek", "do_sample": False, "temperature": 0.0, "max_attempts": 10})

In [2]:
# Discovery
thres = 0.4
model_name = 'StageNet'
sample_dl_candidates = get_dl_candidates(thres, os.path.join(cache_dir, "sensitivity"), dataset, task, full_set, test_set, model_name)
sample_candidate_ranks = rank_dl_candidates(os.path.join(cache_dir, "sensitivity"), dataset, task, full_set, test_set, model_name)

In [3]:
sample_solutions = []

for i in range(len(test_set)):
    patient = test_set[i]
    dl_candidates = sample_dl_candidates[i]
    conditions, procedures, drugs = get_concepts(patient, task, with_history=True, flatten=False)

    solutions_visits = []
    for vid in range(len(conditions)):
        solution = {
            "header": None,
            "conditions": conditions[vid],
            "procedures": procedures[vid],
            "drugs": drugs[vid],
            "candidates": dl_candidates,
            "dl_candidates": dl_candidates
        }
        solutions = [solution]
        solutions_visits.append(solutions)

    sample_solutions.append({
        "solutions": solutions_visits,
        label_key: patient[label_key],
        "isolates": []
    })

In [4]:
from TherapyReason.overview import post_process_solution_overview

template_solution_overview = """\
# Task: Patient Overview
Go over the key information in the patient's context. Summarize the patient's context in one paragraph briefly.

## Patient's Context
{solution_string}

## Output Format
Patient Overview: [Your output]
"""

def solution2string(solution):
    solution_name, solution_path = "", ""
    if solution['conditions']:
        solution_name += "### Conditions\n" + "".join(x + "\n" for x in solution['conditions'])
    if solution['procedures']:
        solution_name += "### Procedures\n" + "".join(x + "\n" for x in solution['procedures'])
    if solution['drugs']:
        solution_name += "### Medications\n" + "".join(x + "\n" for x in solution['drugs'])
    return solution_name

def process_solution_overview_wrap(cache_dir, llm):
    def process_solution_overview(key, item, **kwargs):
        cache_file = Path(os.path.join(cache_dir, f"{key}.json"))
        os.makedirs(cache_file.parent, exist_ok=True)
        if cache_file.exists():
            cached = json.loads(cache_file.read_text(encoding="utf-8"))
        else:
            time.sleep(random.randint(1, 10))
            solution_obj = item

            cached = []
            for visit_id, visit in enumerate(solution_obj['solutions']):
                cached_visit = []
                for solution_id, solution in enumerate(visit):
                    assert solution_id == 0
                    query = template_solution_overview.format(solution_string=solution2string(solution))
                    _, overview, _, usage = llm.understand(query, post_process=post_process_solution_overview, **kwargs)
                    cached_visit.append({"query": query, "overview": overview, "usage": usage.dict()})
                cached.append(cached_visit)
            cache_file.write_text(json.dumps(cached, indent=4), encoding="utf-8")

        return key, cached

    return process_solution_overview

In [5]:
# solution overview
process_solution_overview = process_solution_overview_wrap(os.path.join(cache_dir, "ablation", f"solution_thres{int(thres * 10)}_nokg_overview", llm.genre), llm)
solution_overview_output = multiprocess_call(process_solution_overview, sample_solutions, n_workers=16)

## solution reasoning

In [6]:
# 初始化solution reasoning的cache文件（写空文件）
n_shot = 2
chunk_size = 20

for key, item in enumerate(list(zip(sample_solutions, solution_overview_output))):
    solution_obj, solution_overview = item

    cache_dl_file = Path(os.path.join(cache_dir, "ablation", f"solution_thres{int(thres * 10)}_nokg_dl", llm.genre, f"{key}.json"))
    os.makedirs(os.path.dirname(cache_dl_file), exist_ok=True)
    if not cache_dl_file.exists():
        cached = []
        for visit_id, visit in enumerate(solution_obj['solutions']):
            _cached = []
            for solution_id, solution in enumerate(visit):
                n_chunks = int(np.ceil(len(solution["dl_candidates"]) / chunk_size))
                _cached.append({"dl_candidates": solution["dl_candidates"], "dl_query": [None] * n_chunks, "dl_response": [None] * n_chunks,
                    "dl_prediction": {}, "dl_usage": Usage().dict()})
            cached.append(_cached)
        cache_dl_file.write_text(json.dumps(cached, indent=4), encoding="utf-8")
        print(f"Init {cache_dl_file}")

In [7]:
# 对DL进行reasoning
template_solution_system_diag="""\
# Task: Next-Visit Disease Prediction
Predict the likelihood of specific diseases or health conditions that may manifest or progress by the patient's next medical visit \
based on their current health status. Use available patient data to assess risk factors and early warning signs.
Scores: 0 (Very Unlikely), 1 (Unlikely), 2 (Possible), 3 (Likely), 4 (Very Likely)

## Key Considerations
Conditions:
- Existing Diagnoses: Current diseases/conditions that may worsen or trigger complications (e.g., diabetes increasing infection risk).
- Progression Indicators: Biomarkers, symptoms, or trends suggesting disease advancement (e.g., rising HbA1c, worsening shortness of breath).
- Comorbidities: Interactions between conditions that amplify risk (e.g., hypertension + CKD increasing cardiovascular risk).
- Early Signs: Subtle symptoms or lab abnormalities hinting at new-onset conditions (e.g., fatigue + weight loss suggesting thyroid dysfunction).
Procedures:
- Recent Interventions: Surgeries or treatments that may lead to complications (e.g., post-op infection risk, stent thrombosis).
- Monitoring Gaps: Lack of follow-up tests or imaging that could mask disease progression (e.g., missed colonoscopy for high-risk cancer screening).
- Invasive Devices: Implants/lines increasing infection or thrombosis risk (e.g., central line-associated bloodstream infections).
Medications:
- Adverse Effects: Drugs known to cause conditions (e.g., steroids → hyperglycemia, PPIs → C. diff infection).
- Non-Adherence: Missed doses leading to uncontrolled chronic diseases (e.g., antihypertensives → rebound hypertension).
- Polypharmacy Risks Interactions or cumulative side effects (e.g., NSAIDs + anticoagulants → GI bleeding).
"""

def candidate2string_diag(candidates, solution):
    candidate_string_dict = {node: [] for node in candidates}
    return dict(candidate_string_dict)


def process_solutions_dl(key, item, **kwargs):
    solution_obj, solution_overview = item
    visits = solution_obj['solutions']

    cache_file = Path(os.path.join(cache_dir, "ablation", f"solution_thres{int(thres * 10)}_nokg_dl", llm.genre, f"{key}.json"))
    cached = json.loads(cache_file.read_text(encoding="utf-8"))
    for visit_id, visit in enumerate(visits):
        for solution_id, solution in enumerate(visit):
            overview = solution_overview[visit_id][solution_id]['overview']

            n_chunks = int(np.ceil(len(solution['dl_candidates']) / chunk_size).item())
            for i in range(n_chunks):
                if cached[visit_id][solution_id]['dl_response'][i] is not None: continue

                time.sleep(random.randint(0, 4))

                candidates = solution["dl_candidates"][i * chunk_size: (i + 1) * chunk_size]
                candidate_string_dict = candidate2string_diag(candidates, solution)
                post_process_solution = post_process_solution_wrapper_diag(label_tokenizer, candidates=candidates)
                query = template_solution_dl_diag.format(solution_string=solution2string_with_overview_diag(solution, overview, overview_only=True), candidate_dict=json.dumps(candidate_string_dict, indent=2), candidates="\t".join(candidates))
                _, (response, prediction), _, usage = llm.understand(query, system=template_solution_system_diag, post_process=post_process_solution, **kwargs)

                print(f"Patient: {key}\tVisit: {visit_id + 1}/{len(visits)}\tSolution: {solution_id + 1}/{len(visit)}\tDL Chunk: {i}/{n_chunks}")
                cached[visit_id][solution_id]['dl_query'][i] = query
                cached[visit_id][solution_id]['dl_response'][i] = response
                cached[visit_id][solution_id]['dl_prediction'].update(prediction)
                cached[visit_id][solution_id]['dl_usage'] = (usage + cached[visit_id][solution_id]['dl_usage']).dict()
                cache_file.write_text(json.dumps(cached, indent=4), encoding="utf-8")

    return key, cached


sample_dl_reasoning = multiprocess_call(process_solutions_dl, list(zip(sample_solutions, solution_overview_output)), n_workers=8, verbose=False)

# eval

In [9]:
# 评估DL的结果
def get_dl_metric(offset, threshold, dl_reasoning, agg='mean'):
    y_true_all, y_prob_all = [], []
    for i in range(len(test_set)):
        patient_prediction = defaultdict(list)
        for visit_id, visit in enumerate(dl_reasoning[i]):
            for solution_id in range(len(visit)):
                for entity, value in dl_reasoning[i][visit_id][solution_id]['dl_prediction'].items():
                    assert entity is not None, i
                    patient_prediction[entity].append(value['predict'] + offset)

        patient_prediction = {entity: np.mean(scores) if agg == 'mean' else np.max(scores) for entity, scores in patient_prediction.items()}
        patient_prediction = [entity for entity, score in patient_prediction.items() if score >= threshold]
        y_true_all.append(prepare_labels(task, [test_set[i][label_key]], label_tokenizer)[0].numpy())
        y_prob_all.append(get_logit(task, list(patient_prediction), label_tokenizer))

    metrics = get_metrics_fn(task)(np.stack(y_true_all), np.array(y_prob_all))
    print(offset, threshold, agg, metrics["precision_samples"], metrics["recall_samples"], metrics["f1_samples"], metrics["jaccard_samples"], metrics['hamming_loss'], metrics['rank'])

for agg in ['max', 'mean']:
    for offset in range(2, 4):
        for threshold in range(offset + 1, min(offset + 5, 5)):
            get_dl_metric(offset, threshold, sample_dl_reasoning, agg=agg)